# EX — RNN Real-World Exercise: Sequence Classification

**Goal:** use an LSTM to classify whether a numeric sequence is trending "up" or "down" —
a simplified stand-in for real sequence/time-series classification tasks.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
np.random.seed(0)
seq_len = 20

def make_seq(trend):
    base = np.linspace(0, 1, seq_len) * trend
    noise = np.random.normal(0, 0.15, seq_len)
    return (base + noise).astype(np.float32)

n_per_class = 300
seqs, labels = [], []
for _ in range(n_per_class):
    seqs.append(make_seq(1)); labels.append(1)   # upward trend
    seqs.append(make_seq(-1)); labels.append(0)  # downward trend

X = torch.tensor(np.stack(seqs))[:, :, None]  # (N, seq_len, 1) -- (batch, time, features)
y = torch.tensor(labels)
idx = np.random.permutation(len(X)); X, y = X[idx], y[idx]
split = int(len(X)*0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
print(X.shape)


### TODO 1
Define an `LSTM(input_size=1, hidden_size=16, batch_first=True)` followed by a `Linear(16, 2)` classifier that uses the **final** hidden state.

In [ ]:
class TinyRNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: self.lstm, self.classifier
        pass
    def forward(self, x):
        # TODO: run lstm, take final hidden state, pass through classifier
        pass


<details><summary>Solution</summary>

```python
class TinyRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=16, batch_first=True)
        self.classifier = nn.Linear(16, 2)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        final_hidden = h_n[-1]          # shape (batch, hidden_size)
        return self.classifier(final_hidden)
```
</details>


In [ ]:
class TinyRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=16, batch_first=True)
        self.classifier = nn.Linear(16, 2)
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        return self.classifier(h_n[-1])

model = TinyRNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(8):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f"epoch {epoch}: loss={total_loss/len(X_train):.4f}")

model.eval()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1)
    acc = (preds == y_test).float().mean().item()
print("test accuracy:", acc)


## Key Takeaways
- LSTMs carry a hidden state across timesteps — the final hidden state is a summary of the whole sequence.
- `batch_first=True` keeps tensor shape as (batch, time, features), which is usually more intuitive.
- For real time-series work, also try feeding the full `out` sequence into an attention layer or pooling instead of only the last hidden state.
